In [1]:
import pandas as pd

# Load your uploaded CSV
df = pd.read_csv("agnewstest (1).csv")

# Display first few rows
print(df.head())

# AG News label mapping
label_names = {
    1: "World",
    2: "Sports",
    3: "Business",
    4 : "Sci/Tech"
  }

# Use a subset for faster execution
sample_size = 3000

documents = (
    df["Title"].fillna("") + ". " +
    df["Description"].fillna("")
).tolist()[:sample_size]

true_labels = df["Class Index"].tolist()[:sample_size]

print(f"Number of documents: {len(documents)}")

print("\nExample document:")
print(documents[0][:300])

print("\nLabels:")
for key, value in label_names.items():
    print(f"{key}: {value}")

   Class Index                                              Title  \
0            3                  Fears for T N pension after talks   
1            4  The Race is On: Second Private Team Sets Launc...   
2            4      Ky. Company Wins Grant to Study Peptides (AP)   
3            4      Prediction Unit Helps Forecast Wildfires (AP)   
4            4        Calif. Aims to Limit Farm-Related Smog (AP)   

                                         Description  
0  Unions representing workers at Turner   Newall...  
1  SPACE.com - TORONTO, Canada -- A second\team o...  
2  AP - A company founded by a chemistry research...  
3  AP - It's barely dawn when Mike Fitzpatrick st...  
4  AP - Southern California's smog-fighting agenc...  
Number of documents: 3000

Example document:
Fears for T N pension after talks. Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.

Labels:
1: World
2: Sports
3: Business
4: Sci/

In [2]:
# ============================================================
# SENTENCE-BERT EMBEDDINGS
# ============================================================
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    documents,
    show_progress_bar=True,
    batch_size=64
)

print("Embedding shape:", embeddings.shape)

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
from sklearn.preprocessing import normalize

embeddings = normalize(embeddings)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)

embeddings_pca = pca.fit_transform(embeddings)

In [ ]:
# ============================================================
# KMEANS CLUSTERING
# ============================================================
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import seaborn as sns
n_clusters = 4

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=20
)

cluster_labels = kmeans.fit_predict(embeddings_pca)

print("Clustering completed")
import pandas as pd
cluster_counts=pd.Series(cluster_labels).value_counts()
print("cluster sizes")
print(cluster_counts)
plt.figure(figsize=(8,5))

sns.countplot(x=cluster_counts)

plt.xlabel("Cluster")
plt.ylabel("Number of Documents")
plt.title("Documents per Cluster")

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
#Cluster membership by label
ct = pd.crosstab(
    cluster_labels,
    true_labels
)

plt.figure(figsize=(8,6))

sns.heatmap(
    ct,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("True Label")
plt.ylabel("Cluster")
plt.title("Cluster Membership by True Label")

plt.show()

In [ ]:
# ============================================================
# EVALUATE CLUSTER QUALITY
# ============================================================
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
sil_score = silhouette_score(
    embeddings_pca,
    cluster_labels
)

ch_score = calinski_harabasz_score(
    embeddings_pca,
    cluster_labels
)

db_score = davies_bouldin_score(
    embeddings_pca,
    cluster_labels
)

print("\nCluster Evaluation Metrics")
print("-" * 40)
print(f"Silhouette Score      : {sil_score:.3f}")
print(f"Calinski-Harabasz     : {ch_score:.2f}")
print(f"Davies-Bouldin Index  : {db_score:.3f}")

In [ ]:
# ============================================================
# DISPLAY DOCUMENTS IN EACH CLUSTER
# ============================================================

df = pd.DataFrame({
    "Document": documents,
    "Cluster": cluster_labels
})

for cluster in sorted(df.Cluster.unique()):

    print("\n" + "="*70)
    print(f"CLUSTER {cluster}")
    print("="*70)

    sample_docs = df[df.Cluster == cluster]\
                    .head(5)["Document"]

    for i, doc in enumerate(sample_docs):

        print(f"\nDocument {i+1}")

        preview = doc.replace("\n", " ")

        print(preview[:300])

In [ ]:
plot_df = pd.DataFrame({
    "PC1": embeddings_pca[:, 0],
    "PC2": embeddings_pca[:, 1]
})
label_map = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech"
 }
plot_df["TrueLabel"] = [
    label_map[x]
    for x in true_labels
]

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=plot_df,
    x="PC1",
    y="PC2",
    hue="TrueLabel",
    palette="tab10",
    alpha=0.75
)

plt.title("Document Categories (PCA)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.show()